# External temporal control transfer, fixed protocol 23

Fit only public Metrica Games 1/2 with conservative event-derived control labels.
Game 3 is reserved for a single transfer gate after the fixed final epoch.
No target-match observations, annotations, media or reference keys are attached.
Conditional external metrics do not count as target-match spell F1.
Acknowledgement and source provenance are in EXTERNAL_PRETRAINING.md.


In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys,torch
assert torch.cuda.is_available()
print('GPU:',torch.cuda.get_device_name(0),flush=True)
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working/possession_external_control');WORK.mkdir(parents=True,exist_ok=True)
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
configs=list(INPUT.rglob('control_input.json'));assert len(configs)==1
FILES=configs[0].parent;plan=json.loads(configs[0].read_text())
roots=[p.parent for p in INPUT.rglob('F1_TARGET.md') if (p.parent/'external_possession/temporal_control.py').is_file()];assert len(roots)==1
CODE=roots[0];assert sha(CODE/'F1_TARGET.md')==plan['protocol_sha256']
for relative,digest in plan['source_sha256'].items():assert sha(CODE/relative)==digest
for game in (1,2,3):assert sha(FILES/f'game_{game}.npz')==plan['feature_sha256'][str(game)]
DEPS=Path('/tmp/barca_control_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),PYTHONDONTWRITEBYTECODE='1',HF_HUB_OFFLINE='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
def run(*args):subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=env,check=True)
run('-m','unittest','discover','-s','tests','-v')
print('Full suite passed; starting fixed external control fit.',flush=True)


In [ ]:
run('-m','external_possession.temporal_control','train','--features',FILES,'--out',WORK/'study','--device','cuda')
report=json.loads((WORK/'study/report.json').read_text())
assert report['target_labels_used'] is False
protocol=dict(git_checkpoint=plan['git_checkpoint'],protocol_sha256=plan['protocol_sha256'],source_sha256=plan['source_sha256'],
    feature_sha256=plan['feature_sha256'],torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
    target_match_labels_attached=False,target_match_observations_attached=False,model_sha256=sha(WORK/'study/control.pt'))
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print('EXTERNAL CONTROL GPU RUN COMPLETE',flush=True)
